In [1]:
import pandas as pd
import numpy as np
import genesis as gs

import torch
from rsl_rl.algorithms import PPO
# from rsl_rl.modules.actor_critic import ActorCritic
from rsl_rl.runners import OnPolicyRunner
from rsl_rl.runners import OnPolicyRunner

from tensordict import TensorDict



gs.init(backend=gs.gpu)

[I 11/01/26 16:27:03.898 2953] [shell.py:_shell_pop_print@25] Graphical python shell detected, using wrapped sys.stdout


[Genesis] [16:27:16] [INFO] ╭───────────────────────────────────────────────╮
[Genesis] [16:27:16] [INFO] │┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈ Genesis ┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈┉┈│
[Genesis] [16:27:16] [INFO] ╰───────────────────────────────────────────────╯
[Genesis] [16:27:16] [INFO] Running on [NVIDIA GeForce RTX 3070] with backend gs.cuda. Device memory: 8.00 GB.
[Genesis] [16:27:16] [INFO] 🚀 Genesis initialized. 🔖 version: 1.4.5, 🎨 theme: dark, 🌱 seed: None, 🐛 debug: False, 📏 precision: 32, 🔥 performance: False, 💬 verbose: INFO


In [2]:
# CPUでもgensis回るはずー: これは回った

# robot制御に特化した, RLを使って学習を行う


# ミニマムタスク: RL-歩かせる

- 終了判定
    - 20m進んだら終わる
- 報酬:前に進む
    - その場にとどまることをペナルティーとして与える

- 目的：
    - ロボットの制御 using rlのイメージを作りにいく
    - rlでできそうなことをちゃんと整理する
    - 何が必要か？とか
        - 今のイメージ：関節角 - 目標関節角 => 移動を繰り返す感じになると思う

In [17]:
#一旦ここで、RL回す準備する
# - 観測・行動・報酬のサイクルが回る必要がある
# 何が必要なのか？何の対応が必要なのかイメージで切るようにする

# genesis-wolrdってのが、複数環境を並列で動かせるオモロ仕組みらしい
import genesis as gs
import torch
class Environment: 
    def __init__(self, num_env=1, device="cuda"):
        self.num_envs = num_env
        self.device = device
        
    
        # シーン構築
        self.scene = gs.Scene(
            sim_options= gs.options.SimOptions(dt=0.05, substeps=2), # 本来は細かくした方がいいかもしれんが、メモリが足りない
            show_viewer=False,
        )

        self.plane = self.scene.add_entity(gs.morphs.Plane())
        self.robot = self.scene.add_entity(gs.morphs.URDF(
            file="urdf/go2/urdf/go2.urdf",
            pos=(0, 0, 0.42))) #少し地面から浮いた状態


        self.cam = self.scene.add_camera(
            res=(1280, 720),
            pos=(3.0, -3.0, 5), # カメラの配置位置 (x, y, z) 
            lookat=(0.0, 0.0, 0.5), # 注視点 (ロボットの少し上など) 
            fov=40, # 視野角 
            GUI=False, # GUI描画はオフ 
            )
        
        # if (device == "cuda" or device=="gpu"):
        # 並列構築したい数を渡す
        self.scene.build(n_envs=self.num_envs)
        # ビルド後, 実態が生成
        self.initial_pos = self.robot.get_pos()
        self.initial_state = self.scene.get_state()
        # デフォルの関節角/PDゲインを設定
        self.n_dofs = self.robot.n_dofs
        self.default_dof_pos = torch.zeros((self.num_envs, self.n_dofs), device=self.device)
        self.robot.set_dofs_kp([20.0] * self.n_dofs)
        self.robot.set_dofs_kv([0.5] * self.n_dofs)

        # 観測・行動がもつ次元数の定義
        self.num_obs = self.n_dofs * 2 + 2
        self.num_actions = self.n_dofs # 関節角への指示　＝＝行動
        self.goal_pos = [20, 0, 0]


        # stepで学習やターゲットに使用したデータを格納する
        self.logs ={
            "target_pos": [],
            "rewards": [],
            "observation": [],
            "is_done": [] #終了判定
        }
        self.cfg = {
                    "num_envs": self.num_envs,
                    "device": self.device,
                    "default_position": self.default_dof_pos
                }
        return None

    def reset(self):
        self.scene.reset()
        return self._get_obs()
    
    def steps(self, actions):
        actions = actions.to(self.device)
        target_pos = self.default_dof_pos + actions
        self.robot.control_dofs_position(target_pos)
        self.logs["target_pos"].append(target_pos)
        
        # 物理時間を進める
        self.scene.step()

        # 観測/報酬決定
        obs = self._get_obs()
        rewards = self._compute_rewards()
        dones, fell = self._check_dones()

         # 転倒や目的を達成したepisodeは初期化
        if (dones.numel() > 0):
            self._reset_falling_robot(fell)

        # # step時に,
        # self.cam.set_pose(
        #             pos=self.robot.get_pos().to(self.device) +offset,
        #             lookat=(0.0, 0.0, 0.5),
        #         )

        return obs, rewards, dones

    def _reset_falling_robot(self,fall):

        env_idx = torch.where(fall)[0] #転倒したロボット取得
        if len(env_idx) > 0:
            self.scene.reset(state = self.initial_state , envs_idx=env_idx)
        return None 



    

    def _get_obs(self):
        # ロボットの関節状態
        dofs_position = self.robot.get_dofs_position()
        dofs_velocity = self.robot.get_dofs_velocity()

        current_state = torch.cat([dofs_position, dofs_velocity], dim=-1).to(self.device)

        # image assert self.num_obj.shape == current_state.shape
        return current_state


    def _compute_rewards(self):
        robot_pos = self.robot.get_pos()
        # きめの問題だから、x方向に進んだかどうか？
        # 近づいたかを算出する
        reached_goal = robot_pos[:, 0] >= 20.0
        reward_goal = torch.where(reached_goal, 15, 0) # ゴールした場合のボーナス
        # 通常の報酬
        reward_fwd = self.robot.get_dofs_velocity()[:,0] * 1.0# 速度の方がいいらしい、腹落ち要件

        # +α: 転倒時に罰則を与えるわ => 逆転で頭をあげてられる＝＝姿勢を安定させる
        penelize_falling_idx = robot_pos[:, 2] > 0.35
        stable_posture = torch.where(penelize_falling_idx, 1, 0)
        return reward_fwd+reward_goal+stable_posture


    

    def _check_dones(self):
        # 全環境のロボット位置を取得 
        pos = self.robot.get_pos() 
        #1. X軸方向に20m到達・通過した判定 (Bool Tensor) 
        reached_goal = pos[:, 0] >= 20.0 
        # 2. 転倒判定（例: 胴体の高さが0.2m未満に低下） => 学習した結果が倒れた状態でばたばたになっている
        fell = pos[:, 2] < 0.35
        if (torch.any(fell == True)):
            print("倒れた") 

        # todoこれはせt タイムアウト判定（例: 最大ステップ数を超過） #
        # timeout = self.episode_length_buf >= self.max_episode_length 
        # いずれかの条件を満たした環境を True (終了) にする 

        dones = reached_goal | fell 
        return dones, fell









In [18]:
env = Environment(num_env=25, device="cuda")
obs=env.reset()

print("観測データshape", obs.shape)
FILE_NAME = "20261101_robot_result_2.mp4"
result = {"random_actions":[], "obs":[], "rewards": [], "dones": []}

# 本当に最小限のやつ
# テストで実行
MOVIE_LENGTH = 15
env.cam.start_recording(FILE_NAME, fps=30)
for step in range(int(30*MOVIE_LENGTH+1)):
    random_actions = torch.randn((env.num_envs, env.num_actions))
    obs, rewards, dones =  env.steps(random_actions)
    result["random_actions"].append(random_actions)
    result["obs"].append(obs)
    result["rewards"].append(rewards)
    result["dones"].append(dones)

    print(f"Step{step}: 平均報酬 = {rewards.mean().item()}")

env.cam.stop_recording()



[Genesis] [17:11:16] [INFO] Scene <4ca3f1b> created.
[Genesis] [17:11:16] [INFO] Adding <gs.engine.entities.RigidEntity> plane_1120b26. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.
[Genesis] [17:11:18] [INFO] Adding <gs.engine.entities.RigidEntity> go2_description_e2659ef. idx: 1, morph: <gs.morphs.URDF(file='/usr/local/lib/python3.12/site-packages/genesis/assets/urdf/go2/urdf/go2.urdf')>, material: <gs.materials.Rigid>.
[Genesis] [17:11:18] [INFO] Building scene <4ca3f1b>...
[Genesis] [17:11:18] [WARNING] Neutral robot position (qpos0) exceeds joint limits.
[Genesis] [17:11:19] [INFO] Compiling simulation kernels...
[Genesis] [17:11:19] [INFO] Building visualizer...
[Genesis] [17:11:19] [INFO] Software rendering context detected. Shadows and plane reflection not supported. Beware rendering will be extremely slow.
[Genesis] [17:11:19] [INFO] Raytracer and Rasterizer requires binding to the camera with a specific environment index. Defaulting to 'rendered_envs_idx[0

In [ ]:
# 関節データの形・加速度とかの形どうなってるんだ？


In [ ]:
# RLの部分
    # import genesis as gs
# 1. 先ほど作成した環境クラス
# env = CustomObstacleEnv(num_envs=2048, device="cuda")

# 2. rsl_rl 用の環境情報ラッパーを設定
class RSLRLWrapper:
    def __init__(self, env):
        self.env = env
        self.num_envs = env.num_envs
        self.num_obs = env.num_obs
        self.num_privileged_obs = None
        self.num_actions = env.num_actions
        self.device = env.device
        self.cfg = {"env": self.env, 
                    "num_envs": self.num_envs, 
                    "num_obs":self.num_obs, 
                    "num_actions": self.num_actions}  # 追加


    # def get_observations(self):
    #     return self.env._get_obs(), {}
    def _to_tensordict(self, obs):
        return TensorDict(
            {"policy": obs},
            batch_size=[self.num_envs],
            device=self.device,
        )

    def get_observations(self):
        obs = self.env._get_obs()
        return self._to_tensordict(obs)

    def step(self, actions):
        obs, rewards, dones = self.env.steps(actions)
        obs = self._to_tensordict(obs)
        return obs, rewards, dones, {}

    def reset(self):
        return self.env.reset(), {}







def train(FILE_NAME="20261101_rl_learning_baseline.mp4", num_envs=15):
    # gs.init(backend=gs.gpu)

    raw_env = Environment(num_env=num_envs, device="cuda")
    env = RSLRLWrapper(raw_env)

    train_cfg = {
    # Runner settings
    "num_steps_per_env": 48,
    "save_interval": 100,

    "obs_groups": {
        "actor": ["policy"],
        "critic": ["policy"],
    },

    # PPO settings
    "algorithm": {
        "class_name": "PPO",
        "learning_rate": 1e-3,
        "num_learning_epochs": 5,
        "num_mini_batches": 4,
        "clip_param": 0.2,
        "desired_kl": 0.01,
        "gamma": 0.99,
        "lam": 0.95,
        "entropy_coef": 0.01,
        "value_loss_coef": 1.0,
        "max_grad_norm": 1.0,
    },

    # Actor
    "actor": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
        "distribution_cfg": {
            "class_name": "GaussianDistribution",
            "init_std": 1.0,
        },
    },

    # Critic
    "critic": {
        "class_name": "MLPModel",
        "hidden_dims": [256,128, 64],
        "activation": "elu",
    },}
    ppo_runner = OnPolicyRunner(
        env=env,
        train_cfg=train_cfg,
        # log_dir="logs/go2_rl",
        device=env.device,
    )

    env.env.cam.start_recording(
        FILE_NAME,
        fps=60,
    )
    print("camera-started")

    ppo_runner.learn(
        num_learning_iterations=1500,
        # init_at_random_policy=False,
    )

    env.env.cam.stop_recording()

    return env


In [ ]:
# 学習改善回す
models = train(FILE_NAME="20261102_rl_learning.mp4", num_envs=15)

[Genesis] [17:23:50] [INFO] Scene <f02922b> created.
[Genesis] [17:23:50] [INFO] Adding <gs.engine.entities.RigidEntity> plane_49b1884. idx: 0, morph: <gs.morphs.Plane>, material: <gs.materials.Rigid>.


In [3]:
import rsl_rl

In [38]:
import rsl_rl
print(rsl_rl.__file__)
import importlib.metadata
print(importlib.metadata.version("rsl-rl"))

/usr/local/lib/python3.12/site-packages/rsl_rl/__init__.py


PackageNotFoundError: No package metadata was found for rsl-rl

In [39]:
from rsl_rl.modules import ActorCritic

print(ActorCritic)

ImportError: cannot import name 'ActorCritic' from 'rsl_rl.modules' (/usr/local/lib/python3.12/site-packages/rsl_rl/modules/__init__.py)